<a href="https://colab.research.google.com/github/acastellanos-ie/NLP-MBDS-EN-NEW/blob/main/03_linguistic_structure_and_interpretability/attention.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# The Anatomy of Attention

In the previous session, a dependency parser could not tell what *it* refers to in "The animal didn't cross the street because it was too tired." The grammar is the same whether the animal is tired or the street is wide, yet you, and BERT, resolve the pronoun instantly.

This notebook opens BERT and watches its attention while it reads a sentence. It sits in the interpretability strand of the course: before we trust the model inside the Helcon Support Copilot, we want to see how it routes information. Here we make "understanding context" visible as attention flowing between words.

In [ ]:
# Setup, run this cell first.
# It installs the libraries, pre-downloads BERT (~440 MB) and the small spaCy
# English model, so every later cell loads from cache instantly.
!pip install -q transformers torch spacy matplotlib seaborn numpy
!python -m spacy download en_core_web_sm -q

from huggingface_hub import snapshot_download
snapshot_download("bert-base-uncased")
print("Setup complete.")

## Step 1: The sentence that breaks syntax

We start with last session's tools. The two sentences below differ by exactly one word, *tired* vs *wide*, and that single word flips the referent of *it*: a tired animal, a wide street. Watch what the dependency parser does with the pair.

In [ ]:
import spacy
from spacy import displacy
from IPython.display import display, HTML

nlp = spacy.load("en_core_web_sm")

# Render the dependency tree of each sentence. The arc for 'it' is identical
# in both, so the parser cannot show what the pronoun refers to.
for sentence in ["The animal didn't cross the street because it was too tired.",
                 "The animal didn't cross the street because it was too wide."]:
    doc = nlp(sentence)
    html = displacy.render(doc, style="dep", jupyter=False,
                           options={"distance": 90, "compact": True})
    display(HTML(html))

Two trees, identical arc for arc: *it* is attached exactly the same way in both renders, because grammatically nothing changed. Classical syntax is blind to the one thing that did change, which is what *it* refers to. Yet you resolved both pronouns instantly, and BERT does too. Somewhere inside BERT's 12 layers times 12 heads (144 attention heads in total), that resolution happens. The rest of the session is about finding where.

## Step 2: Loading the model in inspection mode

By default, models discard attention weights after a prediction to save memory. We force the model to return these tensors with `output_attentions=True` so we can read them.

In [ ]:
import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel
import matplotlib.pyplot as plt
import seaborn as sns

model_id = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_id)   # instant, the setup cell already cached it
model = AutoModel.from_pretrained(model_id, output_attentions=True, attn_implementation="eager")
model.eval()

print(f"Loaded model: {model_id} | Parameters: {sum(p.numel() for p in model.parameters()):,}")

## Step 3: Run both sentences through BERT

We feed both sentences to BERT and keep every attention weight for later. Before looking at attention, it helps to see what BERT actually reads: not words, but **tokens**, including the special `[CLS]` and `[SEP]` markers and sub-word pieces for rare words. The cell also checks the one thing the rest of the lab depends on: the two sentences tokenize into the same sequence except for the final word, so the pronoun and both candidate nouns sit at identical positions in A and B. That alignment is what lets us compare the two attention maps position by position.

The experiment is parameterized at the top: the minimal pair, the pronoun, and the two candidate nouns. If you swap in your own pair, keep the two sentences identical except for the final word.

In [ ]:
TEXT_A = "The animal didn't cross the street because it was too tired."
TEXT_B = "The animal didn't cross the street because it was too wide."
PRONOUN     = "it"
CANDIDATE_1 = "animal"   # the referent of the pronoun in sentence A
CANDIDATE_2 = "street"   # the referent of the pronoun in sentence B
# try: "The trophy didn't fit in the suitcase because it was too big." with
#      "... because it was too small.", and CANDIDATE_1 = "trophy", CANDIDATE_2 = "suitcase".
#      Keep the two sentences identical except for the final word.

inputs_a = tokenizer(TEXT_A, return_tensors="pt")
inputs_b = tokenizer(TEXT_B, return_tensors="pt")
tokens_a = tokenizer.convert_ids_to_tokens(inputs_a["input_ids"][0])
tokens_b = tokenizer.convert_ids_to_tokens(inputs_b["input_ids"][0])

# One forward pass per sentence, keeping the attention weights. After this it is pure numpy.
with torch.no_grad():
    outputs_a = model(**inputs_a)
    outputs_b = model(**inputs_b)

# outputs.attentions is a tuple with one tensor per layer, each [batch, heads, tokens, tokens].
# Stack them into one array per sentence: [layers, heads, tokens, tokens].
att_a = np.stack([layer[0].numpy() for layer in outputs_a.attentions])
att_b = np.stack([layer[0].numpy() for layer in outputs_b.attentions])

def token_index(tokens, word):
    """Position of a word in the token list (the uncased model lowercases everything)."""
    word = word.lower()
    if word not in tokens:
        raise ValueError(f"'{word}' is not a single token in: {tokens}")
    return tokens.index(word)

# What BERT actually reads, and the single place the two inputs differ.
print("BERT reads tokens, not words. Each sentence after tokenization:")
print("A:", tokens_a)
print("B:", tokens_b)
differ = [i for i, (x, y) in enumerate(zip(tokens_a, tokens_b)) if x != y]
print()
print(f"The two are identical except position {differ}: "
      f"{[tokens_a[i] for i in differ]} (A) vs {[tokens_b[i] for i in differ]} (B).")
print(f"So '{PRONOUN}', '{CANDIDATE_1}' and '{CANDIDATE_2}' sit at the same positions in both,")
print("which is exactly what lets us compare the two attention maps in the next steps.")
print()
n_layers, n_heads = att_a.shape[0], att_a.shape[1]
print(f"For each sentence BERT produced {n_layers} layers x {n_heads} heads "
      f"= {n_layers * n_heads} attention matrices.")
print(f"Which of those {n_layers * n_heads} heads resolves '{PRONOUN}'? That is Step 4.")

## Step 4: Where does *it* look?

A dependency tree attaches *it* the same way in both sentences. Let's ask BERT a more direct question: of the attention the pronoun *it* pays to our two candidate nouns, how much goes to *animal* and how much to *street*, and does it flip between the two sentences?

Attention is spread across 12 times 12 (144) heads, most of them doing other bookkeeping. We pick the one head whose attention most clearly tracks the pronoun. Among 144 heads, one has learned to do coreference, and we look at what it does.

In [ ]:
# Find the head whose attention most clearly tracks the pronoun, then show what it does.
it_a, c1_a, c2_a = (token_index(tokens_a, w) for w in (PRONOUN, CANDIDATE_1, CANDIDATE_2))
it_b, c1_b, c2_b = (token_index(tokens_b, w) for w in (PRONOUN, CANDIDATE_1, CANDIDATE_2))

# A coreference head prefers CANDIDATE_1 in sentence A and CANDIDATE_2 in sentence B,
# so this "flip" score is large for exactly such a head:
flip = (att_a[:, :, it_a, c1_a] - att_a[:, :, it_a, c2_a]) \
     - (att_b[:, :, it_b, c1_b] - att_b[:, :, it_b, c2_b])
LAYER, HEAD = (int(i) for i in np.unravel_index(np.argmax(flip), flip.shape))
# try: set LAYER, HEAD by hand (e.g. 9, 6, the "BERTology" coreference head) and compare
print(f"Clearest coreference head: layer {LAYER}, head {HEAD}")

# How 'it' splits its attention between the two candidate nouns, in each sentence:
a = [att_a[LAYER, HEAD, it_a, c1_a], att_a[LAYER, HEAD, it_a, c2_a]]
b = [att_b[LAYER, HEAD, it_b, c1_b], att_b[LAYER, HEAD, it_b, c2_b]]
a_share = [100 * x / sum(a) for x in a]   # % of the attention 'it' sends to the two nouns
b_share = [100 * x / sum(b) for x in b]

last_a = TEXT_A.rstrip(".").split()[-1]
last_b = TEXT_B.rstrip(".").split()[-1]
x = np.arange(2)
w = 0.35
plt.figure(figsize=(7, 4.5))
plt.bar(x - w / 2, [a_share[0], b_share[0]], w, label=f"to '{CANDIDATE_1}'")
plt.bar(x + w / 2, [a_share[1], b_share[1]], w, label=f"to '{CANDIDATE_2}'")
plt.xticks(x, [f"Sentence A\n(...too {last_a})", f"Sentence B\n(...too {last_b})"])
plt.ylabel(f"where '{PRONOUN}' looks (% of its attention to the two nouns)")
plt.title("Same pronoun, same grammar - different referent")
plt.ylim(0, 100)
plt.legend()
plt.tight_layout()
plt.show()

print(f"Sentence A: '{PRONOUN}' -> {CANDIDATE_1} {a_share[0]:.0f}%  |  {CANDIDATE_2} {a_share[1]:.0f}%")
print(f"Sentence B: '{PRONOUN}' -> {CANDIDATE_1} {b_share[0]:.0f}%  |  {CANDIDATE_2} {b_share[1]:.0f}%")

## Takeaway

Same word, same grammar, but when the sentence says *tired*, *it* sends most of its attention to *animal*, and when it says *wide*, to *street*. The pronoun's routing flipped because one adjective changed, while the dependency trees of step 1 stayed identical. That flip is coreference being resolved, not by a grammar rule, but by attention routing information between positions. The exact percentages vary by model version. What is stable is that the majority share flips with the context.

## Step 5 (optional): the raw attention matrices

Want to see the actual matrices behind that bar chart? Here are both sentences for the same head, side by side. Two things to know before reading them. The bright columns on `[CLS]`, `[SEP]` and the final period are attention sinks, leftover probability the head parks on always-present tokens, so ignore them. Look at the row for *it* and compare its *animal* and *street* columns across the two matrices.

In [ ]:
# Plot the full attention matrix of the chosen head for each sentence,
# so the row for 'it' can be read directly.
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
panels = [(axes[0], att_a, tokens_a, f"A (...{last_a})"),
          (axes[1], att_b, tokens_b, f"B (...{last_b})")]
for ax, att, toks, label in panels:
    sns.heatmap(att[LAYER, HEAD], xticklabels=toks, yticklabels=toks,
                cmap="viridis", ax=ax, cbar=False)
    ax.set_title(f"Sentence {label}  |  layer {LAYER}, head {HEAD}")
plt.tight_layout()
plt.show()

## Takeaway

We found, without training anything, a place inside BERT where context physically changes what a word attends to, the routing the dependency tree was blind to. One honest caveat for later: we picked the single clearest head out of 144, and different heads disagree. An attention map is a good instrument for debugging and building intuition about what a model routes, but on its own it is not a courtroom-grade explanation of why the model decided something. (Further reading, if curious: *Attention is not Explanation*, Jain & Wallace 2019.)